# PC-FNO3D national — règle GFS v4, partie A (Colab CPU) : GFS f0 / f24 / f48

Règle `configs/national/regle_national_gfs_v4.json` (sha256 `4ca1f2a0c348…`), approuvée par Ines le 16/11/2026 et vérifiée par le carnet. Code : dépôt public au commit `6e1461089660`. Sorties : `MyDrive/pcfno3d_national/gfs_v4_regle/` (dossier neuf). **Seconde lecture** de 2022 (mêmes METAR que le DSO7). GFS et ERA5 ne sont jamais appelés mesures. Aucun réentraînement, aucun bruit ajouté. La session est libérée à la fin.

## 1. Drive, code au commit figé, données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "6e1461089660c48bee6169ccdd17779deb464220"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"), ("regle_national_gfs_v4.json", "4ca1f2a0c34856d6840d620cd9457534c37cb316a59080400e2853e2f0d7d721")):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regles v4 et gfs_v4 verifiees")

GV4 = NAT / "gfs_v4_regle"                 # dossier NEUF de la regle (rien d'autre n'y ecrit)
GV4.mkdir(exist_ok=True)
F = {e: GV4 / f"forcage_gfs_f{e}.csv" for e in (0, 24, 48)}
RES = GV4 / "resultat"
print("accord :", "Ines, 16/11/2026 : « 1 start execution » (regle approuvee telle quelle, puis calcul)")

## 2. GFS (archive NOAA sur AWS), boîte nationale, instants synoptiques du test ; fin de la session

In [ ]:
# PARTIE A (Colab CPU) : GFS a l'analyse, +24 h et +48 h aux instants synoptiques du test 2022, boite nationale
!pip install -q herbie-data cfgrib eccodes 2>&1 | tail -1
from google.colab import runtime
import traceback
try:
    CACHE = Path("/content/national/cache")          # l'index du cache v3 seulement (pas les ~50 Go)
    CACHE.mkdir(parents=True, exist_ok=True)
    import pandas as pd
    parts = []
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            with zf.open(f"index_partie_{k}.csv") as fi:
                parts.append(pd.read_csv(fi))
    pd.concat(parts).drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
    for e in (0, 24, 48):
        if F[e].exists():
            print(f"f{e} deja ecrit : {F[e]}")
            continue
        lecture = f"--lecture {GT / 'os7' / f'gfs_f{e}_instants'}" if e else ""
        # code 4 = plus de 2 % d'echeances absentes de l'archive apres 3 passes (regle) : arret sans verdict
        lancer(f"gfs_v4_f{e}", f"python -u -m pcfno3d.gfs --config national_1km_colab --national --echeance {e} "
                               f"--splits test --synoptiques --sortie {F[e]} --reprise {GV4 / f'gfs_f{e}_tn'} "
                               f"{lecture} --travailleurs 8")
    for e in (0, 24, 48):
        err = F[e].with_suffix(".erreurs.json")
        print(f"f{e} :", F[e].exists(), "| echeances absentes :", len(json.loads(err.read_text())) if err.exists() else 0)
    print("partie A terminee : lancer le carnet B (A100)")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()